# Pipeline 07: SLM Distillation & Edge Export

**Goal:** Fine-tune Qwen 2.5 1.5B to replicate Claude's parsing capabilities. Export to GGUF for offline/mobile inference.

**Depends on:** Pipeline 06 (training dataset)

**Runtime:** GPU required (T4 minimum for QLoRA). A100 recommended for speed.

In [1]:
# Setup
!pip install unsloth -q
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q

from google.colab import drive
drive.mount('/content/drive')

import os, json, torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

PROJECT = '/content/drive/MyDrive/photo-style-rl'
DATASET_PATH = f'{PROJECT}/data/slm_training_dataset.jsonl'
OUTPUT_DIR = f'{PROJECT}/models/qwen_fuji_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

In [2]:
# Data sanitization (strict)
# Same validation from Pipeline 06, applied at training time for defense-in-depth
VALID_TONE_CURVE_KEYS = {"blacks", "shadows", "midtones", "highlights", "whites"}
VALID_COLOR_NAMES = {"reds", "oranges", "yellows", "greens", "aquas", "blues", "purples", "magentas"}
VALID_HSL_KEYS = {"h", "s", "l"}
VALID_BASIC_KEYS = {"exposure", "contrast", "temperature", "shadows", "highlights",
                    "clarity", "vibrance", "saturation", "texture", "dehaze"}

def validate_payload(payload):
    if not isinstance(payload, dict):
        return False
    for region, edits in payload.items():
        if not isinstance(edits, dict):
            return False
        for key, value in edits.items():
            if key == 'tone_curve':
                if not isinstance(value, dict): return False
                for k, v in value.items():
                    if k not in VALID_TONE_CURVE_KEYS: return False
                    if not isinstance(v, (int, float)) or not (-100 <= v <= 100): return False
            elif key == 'color_mixer':
                if not isinstance(value, dict): return False
                for color, shifts in value.items():
                    if color not in VALID_COLOR_NAMES: return False
                    if not isinstance(shifts, dict): return False
                    for axis, val in shifts.items():
                        if axis not in VALID_HSL_KEYS: return False
                        if not isinstance(val, (int, float)): return False
                        limit = 180 if axis == 'h' else 100
                        if not (-limit <= val <= limit): return False
            elif key in VALID_BASIC_KEYS:
                if not isinstance(value, (int, float)): return False
                limit = 5 if key == 'exposure' else 100
                if not (-limit <= value <= limit): return False
            else:
                return False
    return True

# Load and validate
valid_records = []
total = 0
with open(DATASET_PATH, 'r') as f:
    for line in f:
        total += 1
        record = json.loads(line)
        try:
            payload = json.loads(record['messages'][1]['content'])
            if validate_payload(payload):
                valid_records.append(record)
        except:
            pass

print(f"Loaded {len(valid_records)} valid records from {total} total ({total - len(valid_records)} rejected)")

if len(valid_records) < 50:
    print("⚠️ Fewer than 50 clean records. Run Pipeline 06 to generate more data.")

dataset = Dataset.from_list(valid_records)

Loaded 196 valid records from 220 total (24 rejected)


In [3]:
# Model loading + tokenization
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_func(examples):
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)
print(f"Model loaded. Dataset tokenized: {len(dataset)} records.")

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Model loaded. Dataset tokenized: 196 records.


In [4]:
# QLoRA training
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Scale max_steps to dataset size
# Rule of thumb: 2-3 epochs worth of steps
steps_per_epoch = max(len(dataset) // (2 * 4), 1)  # batch_size=2, grad_accum=4
max_steps = steps_per_epoch * 3
print(f"Training for {max_steps} steps (~3 epochs over {len(dataset)} records)")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=max_steps,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting fine-tuning...")
stats = trainer.train()
print(f"Done. Peak GPU memory: {round(torch.cuda.max_memory_reserved() / 1e9, 2)} GB")

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Training for 72 steps (~3 epochs over 196 records)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/196 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 196 | Num Epochs = 3 | Total steps = 72
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,3.242825
2,3.006756
3,3.027911
4,2.603073
5,2.529794
6,2.305875
7,2.033074
8,1.925021
9,1.681055
10,1.685619


Done. Peak GPU memory: 3.01 GB


In [5]:
# Export to GGUF
export_path = f"{OUTPUT_DIR}/qwen_fuji_q4_k_m"

model.save_pretrained_gguf(
    export_path,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\nExported to: {export_path}-unsloth.Q4_K_M.gguf")
print("Drop this into LM Studio, Ollama, or MLX for offline inference.")
print("\nFull pipeline complete:")
print("  01: Text → Params (Claude API)")
print("  02: Style Profiling + SAM Region Detection")
print("  03: Advanced Color Science Extraction")
print("  04: End-to-End Inference Demo")
print("  05: Interactive UI + Data Flywheel")
print("  06: Synthetic Data Distillation")
print("  07: Qwen Fine-Tuning + Edge Export")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 301.55it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:36<00:00, 156.66s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/photo-